# مدل Physics-Informed برای پیش‌بینی IFC و نوار فونون کامل — دیتاست MAX Phase (v3)

**نسخه‌ی تشخیصی + تعادل‌بخشی — بر اساس نتایج v1 و v2**

### خلاصه‌ی تاریخچه

| نسخه | تغییر کلیدی | IFC MSE | ASR Loss | MAE فرکانس |
|---|---|---|---|---|
| v1 | بدون ضریب تبدیل، وزن کم فیزیک | 0.0223 | 1.860 | **17.14** ❌ |
| v2 | + ضریب تبدیل واحد + وزن فیزیک بالا (λ_asr=1.0) | 0.1104 (بدتر!) | 0.0026 (خوب) | **17.28** ❌ (بدون تغییر) |

### نتیجه‌گیری از مقایسه‌ی v1 و v2
دو سرنخ مهم داریم:
1. **اصلاح ضریب تبدیل واحد هیچ تاثیری نداشت** (17.14 → 17.28) — یعنی مشکل از مقیاس فیزیکی نبود؛ جای دیگری است.
2. **افزایش شدید وزن ASR باعث شد IFC MSE تقریباً ۵ برابر بدتر شود** (0.0223 → 0.1104) — این یعنی Loss imbalance: مدل برای راضی کردن قید ASR، از یادگیری مقدار واقعی IFC صرف‌نظر کرده.

### این نسخه (v3) دو کار می‌کند:
1. **سل تشخیصی جدید**: قبل از هر training، توزیع eigenvalueهای ماتریس دینامیکی را — هم برای IFC **واقعی** (Ground Truth) و هم برای یک مدل **دست‌نخورده** (untrained) — بررسی می‌کند. این مشخص می‌کند که آیا مشکل از:
   - خود IFC واقعی Phonopy (شاید این فایل‌ها به‌گونه‌ای هستند که این روش ساده‌ی نقطه-Γ برایشان مودهای ناپایدار زیادی می‌دهد)
   - یا از مدل (پیش‌بینی بد) است.
2. **بازتعادل وزن‌ها**: کاهش λ_asr و λ_symmetry به مقادیر معقول‌تر (نه آنقدر کم که قید نادیده گرفته شود، نه آنقدر زیاد که IFC MSE قربانی شود)


In [ ]:
!pip install -q torch_geometric pymatgen torch-cluster torch-sparse torch-scatter torch-spline-conv -t /kaggle/working/custom_lib
import sys
sys.path.append('/kaggle/working/custom_lib')
print("✅ نصب شد")

In [ ]:
import os
import json
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import Set2Set, global_mean_pool, global_max_pool
from torch_geometric.utils import softmax

from scipy.spatial.distance import cdist
from sklearn.metrics import mean_absolute_error
from tqdm.notebook import tqdm

device = torch.device('cpu')
print(f"Device: {device}")

## 2. مسیرهای داده

In [ ]:
FC_DIR       = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_force_constant_All/extracted_force_constant_C'
POSCAR_DIR   = '/kaggle/input/datasets/metisa81/pgcnndata/all_poscar/all_poscar'
BANDS_DIR    = '/kaggle/input/datasets/metisa81/pgcnndata/extracted_bands_C/extracted_bands_C'
ELASTIC_FILE = '/kaggle/input/d/metisa81/features-dataset/mechanical_data_fixed.json'
FEATURE_CSV  = '/kaggle/input/d/metisa81/feature-dataset-split/5_geometry_radius_volume.csv'

for name, path in [('FC_DIR', FC_DIR), ('POSCAR_DIR', POSCAR_DIR),
                    ('BANDS_DIR', BANDS_DIR), ('ELASTIC_FILE', ELASTIC_FILE),
                    ('FEATURE_CSV', FEATURE_CSV)]:
    exists = '✅' if os.path.exists(path) else '❌ یافت نشد'
    print(f"{exists}  {name}  →  {path}")

## 3. پارسرهای داده

In [ ]:
def parse_poscar(path: str) -> dict:
    with open(path, 'r') as f:
        lines = f.readlines()

    scaling = float(lines[1].strip())
    lattice = np.array(
        [list(map(float, lines[i].split())) for i in range(2, 5)]
    ) * scaling

    line5 = lines[5].strip().split()
    line6 = lines[6].strip().split()

    if line5[0].isalpha():
        elements   = line5
        counts     = list(map(int, line6))
        coord_line = 7
    else:
        counts     = list(map(int, line5))
        elements   = [f'X{i}' for i in range(len(counts))]
        coord_line = 6

    coord_type = lines[coord_line].strip()[0].upper()
    n_atoms    = sum(counts)

    positions = np.array([
        list(map(float, lines[i].split()[:3]))
        for i in range(coord_line + 1, coord_line + 1 + n_atoms)
    ])

    if coord_type == 'D':
        positions = positions @ lattice

    atom_elements = []
    for elem, cnt in zip(elements, counts):
        atom_elements.extend([elem] * cnt)

    return {
        'lattice':       lattice.astype(np.float32),
        'positions':     positions.astype(np.float32),
        'elements':      elements,
        'counts':        counts,
        'atom_elements': atom_elements,
        'n_atoms':       n_atoms,
        'volume':        float(np.abs(np.linalg.det(lattice))),
    }


def parse_force_constants(fc_path: str):
    with open(fc_path, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]

    n_atoms = int(lines[0].split()[0])
    ifcs    = np.zeros((n_atoms, n_atoms, 3, 3), dtype=np.float64)

    idx = 1
    while idx < len(lines):
        parts = lines[idx].split()
        if len(parts) == 2:
            try:
                i, j = int(parts[0]) - 1, int(parts[1]) - 1
                idx += 1
                for row in range(3):
                    vals = list(map(float, lines[idx].split()))
                    ifcs[i, j, row, :] = vals[:3]
                    idx += 1
            except (ValueError, IndexError):
                idx += 1
        else:
            idx += 1

    return n_atoms, ifcs

print("✅ پارسرها آماده شدند")

## 4. ساخت دیتاست خام

In [ ]:
fc_dict     = {Path(f).stem: f for f in Path(FC_DIR).glob('*.fc')}
poscar_dict = {Path(f).stem: f for f in Path(POSCAR_DIR).glob('*.psc')}

band_dict = {}
for f in tqdm(list(Path(BANDS_DIR).glob('*.yaml')), desc="بارگذاری YAML بندها"):
    with open(f, 'r') as file:
        data = yaml.safe_load(file)
        if 'phonon' in data:
            freqs = []
            for point in data['phonon']:
                band_freqs = [b['frequency'] for b in point.get('band', [])]
                if band_freqs:
                    freqs.append(band_freqs)
            if freqs:
                band_dict[f.stem] = np.array(freqs, dtype=np.float32)

common = sorted(set(fc_dict) & set(poscar_dict) & set(band_dict))
print(f"تعداد مواد مشترک: {len(common)}")

shapes = [band_dict[f].shape for f in common]
target_shape = Counter(shapes).most_common(1)[0][0]
print(f"شکل استاندارد نوار فونون (n_qpoints, n_bands): {target_shape}")
N_QPOINTS, N_BANDS = target_shape

with open(ELASTIC_FILE) as f:
    elastic_json = json.load(f)
ELASTIC_KEYS = ['C11','C12','C13','C33','C44','C66',
                'B_Hill','G_Hill','E_Hill','Poisson_Hill',
                'Pugh_ratio','Debye_temperature']
elastic_data = {}
for entry in elastic_json:
    mat = entry.get('material', '')
    vals = [float(entry.get(k, 0) or 0) for k in ELASTIC_KEYS]
    elastic_data[mat] = np.array(vals, dtype=np.float32)

raw_samples = []
failed = []
MAX_ATOMS_FOR_IFC = 12

for formula in tqdm(common, desc="ساخت دیتاست خام"):
    try:
        band_arr = band_dict[formula]
        if band_arr.shape != target_shape:
            continue
        poscar = parse_poscar(str(poscar_dict[formula]))
        n_atoms, ifcs = parse_force_constants(str(fc_dict[formula]))
        if n_atoms != poscar['n_atoms']:
            continue
        if n_atoms > MAX_ATOMS_FOR_IFC:
            continue

        raw_samples.append({
            'formula':           formula,
            'n_atoms':           n_atoms,
            'lattice':           poscar['lattice'],
            'positions':         poscar['positions'],
            'atom_elements':     poscar['atom_elements'],
            'force_constants':   ifcs.astype(np.float32),
            'elastic_constants': elastic_data.get(formula, np.zeros(len(ELASTIC_KEYS), np.float32)),
            'y_phonon':          band_arr.astype(np.float32),
        })
    except Exception as e:
        failed.append((formula, str(e)))

print(f"\n✅ موفق: {len(raw_samples)}  |  ❌ ناموفق/رد شده: {len(failed)}")

## 5. تقسیم Train / Val / Test (seed=42)

In [ ]:
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
idx = rng.permutation(len(raw_samples))

n_total = len(raw_samples)
n_tr = int(0.8 * n_total)
n_va = int(0.1 * n_total)

train_idx = idx[:n_tr]
val_idx   = idx[n_tr:n_tr+n_va]
test_idx  = idx[n_tr+n_va:]

print(f"Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

## 6. ساخت گراف (Bond + Atom)

In [ ]:
CUTOFF = 8.0

df_raw = pd.read_csv(FEATURE_CSV)
symbol_col = next((c for c in df_raw.columns if c.lower() in ['symbol', 'element', 'atom']), df_raw.columns[0])
feature_cols = [c for c in df_raw.columns if c not in [symbol_col, 'atomic_number']]
feature_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
df_features = df_raw[feature_cols].copy()
df_features.index = df_raw[symbol_col]
N_ATOM_FEATURES = len(feature_cols)
print(f"فیچرهای اتمی: {N_ATOM_FEATURES} → {feature_cols}")


def structure_to_bond_graph(positions, n_atoms_sample):
    n = len(positions)
    distances = cdist(positions, positions)
    bonds = [(i, j) for i in range(n) for j in range(i+1, n) if distances[i, j] <= CUTOFF]

    if len(bonds) == 0:
        x = torch.zeros((1, 6), dtype=torch.float)
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 1), dtype=torch.float)
        u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
        return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))

    node_features = []
    for i, j in bonds:
        d = distances[i, j]
        coord_i = np.sum(distances[i] <= CUTOFF) - 1
        coord_j = np.sum(distances[j] <= CUTOFF) - 1
        node_features.append([d, coord_i, coord_j, (coord_i + coord_j) / 2, 0.0, 0.0])
    x = torch.tensor(node_features, dtype=torch.float)

    edge_index, edge_attr = [], []
    for idx1, (i1, j1) in enumerate(bonds):
        for idx2, (i2, j2) in enumerate(bonds[idx1+1:]):
            idx2 += idx1 + 1
            shared = set([i1, j1]) & set([i2, j2])
            if len(shared) == 1:
                s = shared.pop()
                a = positions[i1 if i1 != s else j1] - positions[s]
                b = positions[i2 if i2 != s else j2] - positions[s]
                na, nb = np.linalg.norm(a), np.linalg.norm(b)
                if na > 0 and nb > 0:
                    cos_angle = np.clip(np.dot(a, b) / (na * nb), -1.0, 1.0)
                    angle = np.arccos(cos_angle)
                    edge_index.extend([[idx1, idx2], [idx2, idx1]])
                    edge_attr.extend([[angle], [angle]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous() if edge_index else torch.zeros((2, 0), dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float) if edge_attr else torch.zeros((0, 1), dtype=torch.float)
    u = torch.tensor([[1.0, 1.0, 1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))


def structure_to_atom_graph(atom_elements, positions, n_atoms_sample):
    node_features = []
    for elem in atom_elements:
        if elem in df_features.index:
            feats = df_features.loc[elem].values.astype(np.float32)
        else:
            feats = np.zeros(N_ATOM_FEATURES, dtype=np.float32)
        node_features.append(feats)
    x = torch.tensor(np.array(node_features), dtype=torch.float)

    distances = cdist(positions, positions)
    edge_index, edge_attr = [], []
    n = len(positions)
    for i in range(n):
        for j in range(n):
            if i != j and distances[i, j] <= CUTOFF:
                edge_index.append([i, j])
                edge_attr.append([distances[i, j]])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    u = torch.tensor([[1.0]], dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr, u=u, n_atoms=torch.tensor([n_atoms_sample]))

print("✅ توابع ساخت گراف آماده شدند")

In [ ]:
bond_graphs, atom_graphs, ifc_targets, phonon_targets = [], [], [], []

for s in tqdm(raw_samples, desc="ساخت گراف‌ها"):
    bond_graphs.append(structure_to_bond_graph(s['positions'], s['n_atoms']))
    atom_graphs.append(structure_to_atom_graph(s['atom_elements'], s['positions'], s['n_atoms']))
    ifc_targets.append(s['force_constants'])
    phonon_targets.append(s['y_phonon'])

print(f"✅ {len(bond_graphs)} نمونه گراف‌سازی شد")

## 7. معماری مدل: Dual Graph GNN → پیش‌بینی IFC کامل (بدون تغییر نسبت به v2)

و ثابت تبدیل واحد + توابع محاسبه‌ی فرکانس (حفظ‌شده از v2).

In [ ]:
MAX_ATOMS = max(s['n_atoms'] for s in raw_samples)
print(f"حداکثر تعداد اتم در دیتاست: {MAX_ATOMS}")

In [ ]:
class CustomMessagePassing(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.attention_mlp = nn.Sequential(
            nn.Linear(3 * hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, 1),
            nn.LeakyReLU(0.2)
        )
        self.message_mlp = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
        )

    def forward(self, x, edge_index, edge_attr):
        source_nodes = edge_index[0]
        target_nodes = edge_index[1]
        neighbor_features = x[source_nodes]
        target_features = x[target_nodes]

        attention_input = torch.cat([target_features, neighbor_features, edge_attr], dim=1)
        attention_scores = self.attention_mlp(attention_input)
        attention_weights = softmax(attention_scores, target_nodes, num_nodes=x.size(0))

        messages = neighbor_features * edge_attr
        weighted_messages = messages * attention_weights

        num_nodes = x.size(0)
        aggregated = torch.zeros(num_nodes, self.hidden_dim, device=x.device)
        aggregated.index_add_(0, target_nodes, weighted_messages)

        return self.message_mlp(aggregated)


class DualGraphIFCNet(nn.Module):
    def __init__(self, n_bond_features=6, n_atom_features=7, edge_dim=1, max_atoms=12):
        super().__init__()
        self.max_atoms = max_atoms
        hidden = 128 if n_atom_features <= 10 else 96

        self.bond_embedding = nn.Sequential(
            nn.Linear(n_bond_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.bond_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.bond_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(5)])
        self.bond_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(5)])
        self.bond_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.atom_embedding = nn.Sequential(
            nn.Linear(n_atom_features, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(0.1))
        self.atom_edge_embedding = nn.Sequential(nn.Linear(edge_dim, hidden), nn.SiLU())
        self.atom_message_layers = nn.ModuleList([CustomMessagePassing(hidden) for _ in range(2)])
        self.atom_layer_norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(2)])
        self.atom_attention = nn.Sequential(
            nn.Linear(hidden, hidden // 4), nn.SiLU(), nn.Linear(hidden // 4, 1), nn.Sigmoid())

        self.bond_residual_weight = nn.Parameter(torch.tensor(0.3))
        self.atom_residual_weight = nn.Parameter(torch.tensor(0.3))

        self.set2set_pool = Set2Set(hidden, processing_steps=1)
        self.mean_pool = global_mean_pool
        self.max_pool = global_max_pool
        self.global_mlp = nn.Sequential(nn.Linear(3, hidden // 4), nn.SiLU())

        combined_dim = 8 * hidden + hidden // 4  # اصلاح متیسا
        ifc_out_dim = max_atoms * max_atoms * 3 * 3

        self.ifc_head = nn.Sequential(
            nn.Linear(combined_dim, 512), nn.LayerNorm(512), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.LayerNorm(256), nn.SiLU(), nn.Dropout(0.2),
            nn.Linear(256, ifc_out_dim)
        )

    def _encode_graph(self, x, edge_index, edge_attr, embedding, edge_embedding,
                       message_layers, layer_norms, attention, residual_weight):
        x = embedding(x)
        edge_feat = edge_embedding(edge_attr)
        for i, (msg, ln) in enumerate(zip(message_layers, layer_norms)):
            x_res = x
            x = msg(x, edge_index, edge_feat)
            x = ln(x)
            if i > 0:
                x = x + residual_weight * x_res
            x = F.silu(x)
        attn = attention(x)
        return x * attn, x

    def forward(self, bond_data, atom_data):
        bond_w, x_bond = self._encode_graph(
            bond_data.x, bond_data.edge_index, bond_data.edge_attr,
            self.bond_embedding, self.bond_edge_embedding,
            self.bond_message_layers, self.bond_layer_norms,
            self.bond_attention, self.bond_residual_weight)

        bond_set2set = self.set2set_pool(bond_w, bond_data.batch)
        bond_mean = self.mean_pool(x_bond, bond_data.batch)
        bond_max = self.max_pool(x_bond, bond_data.batch)

        atom_w, x_atom = self._encode_graph(
            atom_data.x, atom_data.edge_index, atom_data.edge_attr,
            self.atom_embedding, self.atom_edge_embedding,
            self.atom_message_layers, self.atom_layer_norms,
            self.atom_attention, self.atom_residual_weight)

        atom_set2set = self.set2set_pool(atom_w, atom_data.batch)
        atom_mean = self.mean_pool(x_atom, atom_data.batch)
        atom_max = self.max_pool(x_atom, atom_data.batch)

        global_features = self.global_mlp(bond_data.u)

        zero_pad = torch.zeros_like(bond_mean)
        combined = torch.cat([
            bond_set2set, bond_mean, bond_max, zero_pad,
            atom_set2set, atom_mean, atom_max, zero_pad,
            global_features
        ], dim=1)

        ifc_flat = self.ifc_head(combined)
        batch_size = ifc_flat.shape[0]
        ifc_pred = ifc_flat.view(batch_size, self.max_atoms, self.max_atoms, 3, 3)
        return ifc_pred

print("✅ معماری DualGraphIFCNet تعریف شد")

In [ ]:
EV_ANGSTROM2_AMU_TO_THZ2 = 15.633302

def compute_phonon_frequencies_gamma(ifc, masses, n_atoms_real):
    n = n_atoms_real
    ifc_real = ifc[:n, :n, :, :]
    mass_factor = torch.sqrt(masses[:n].unsqueeze(1) * masses[:n].unsqueeze(0))
    dynmat = ifc_real / mass_factor.unsqueeze(-1).unsqueeze(-1)
    dynmat_2d = dynmat.permute(0, 2, 1, 3).reshape(3 * n, 3 * n)
    dynmat_2d = 0.5 * (dynmat_2d + dynmat_2d.T)
    eigenvalues = torch.linalg.eigvalsh(dynmat_2d)
    eigenvalues_thz2 = eigenvalues * EV_ANGSTROM2_AMU_TO_THZ2
    omega = torch.sign(eigenvalues_thz2) * torch.sqrt(torch.abs(eigenvalues_thz2) + 1e-8)
    frequencies_thz = omega / (2 * np.pi)
    return frequencies_thz


def compute_eigenvalues_raw(ifc, masses, n_atoms_real):
    """نسخه‌ی خام (بدون تبدیل واحد) برای دیباگ — eigenvalueهای ماتریس دینامیکی."""
    n = n_atoms_real
    ifc_real = ifc[:n, :n, :, :]
    mass_factor = torch.sqrt(masses[:n].unsqueeze(1) * masses[:n].unsqueeze(0))
    dynmat = ifc_real / mass_factor.unsqueeze(-1).unsqueeze(-1)
    dynmat_2d = dynmat.permute(0, 2, 1, 3).reshape(3 * n, 3 * n)
    dynmat_2d = 0.5 * (dynmat_2d + dynmat_2d.T)
    eigenvalues = torch.linalg.eigvalsh(dynmat_2d)
    return eigenvalues


ATOMIC_MASSES = {
    'Ti': 47.87, 'V': 50.94, 'Cr': 52.00, 'Zr': 91.22, 'Nb': 92.91,
    'Mo': 95.95, 'Hf': 178.49, 'Ta': 180.95, 'W': 183.84, 'Sc': 44.96,
    'Al': 26.98, 'Si': 28.09, 'Ga': 69.72, 'Ge': 72.63, 'In': 114.82,
    'Sn': 118.71, 'P': 30.97, 'As': 74.92, 'S': 32.07, 'C': 12.01,
    'N': 14.01,
}

def get_masses_tensor(atom_elements, max_atoms):
    masses = torch.ones(max_atoms, dtype=torch.float32) * 50.0
    for i, elem in enumerate(atom_elements):
        masses[i] = ATOMIC_MASSES.get(elem, 50.0)
    return masses

print("✅ توابع آماده شدند")

## 8. 🔬 سل تشخیصی — بررسی Eigenvalueها قبل از training

این سل، **با استفاده از معماری بالا** (که حالا تعریف شده)، توزیع eigenvalueهای ماتریس دینامیکی را — هم برای IFC **واقعی** (Ground Truth) و هم برای یک مدل **دست‌نخورده** (untrained) — بررسی می‌کند.

**هدف:** اگر حتی با IFC واقعی، فرکانس محاسبه‌شده با `y_phonon` همخوانی نداشت، یعنی مشکل از نحوه‌ی محاسبه‌ی ما (فرمول Dynamical Matrix یا فرض نقطه Γ) است — **نه از مدل**.

In [ ]:
print("="*70)
print("🔬 تست ۱: محاسبه فرکانس از IFC واقعی (Ground Truth) — باید نزدیک به y_phonon باشد")
print("="*70)

sanity_errors = []
n_negative_modes_list = []

for i in test_idx[:20]:  # نمونه‌ای از تست برای سرعت
    s = raw_samples[i]
    n = s['n_atoms']
    ifc_true = torch.tensor(s['force_constants'], dtype=torch.float32)
    masses = get_masses_tensor(s['atom_elements'], n)  # دقیقاً اندازه n (نه MAX_ATOMS) برای این تست

    eigvals_raw = compute_eigenvalues_raw(ifc_true, masses, n)
    n_negative = (eigvals_raw < 0).sum().item()
    n_negative_modes_list.append(n_negative)

    freqs = compute_phonon_frequencies_gamma(ifc_true, masses, n)
    true_peak = float(np.max(s['y_phonon']))
    pred_peak_from_true_ifc = float(torch.max(torch.abs(freqs)).item())
    error = abs(true_peak - pred_peak_from_true_ifc)
    sanity_errors.append(error)

    print(f"{s['formula']:10s} | n_atoms={n:2d} | مودهای منفی={n_negative:2d}/{3*n} | "
          f"peak واقعی={true_peak:7.2f} THz | peak از IFC واقعی={pred_peak_from_true_ifc:7.2f} THz | خطا={error:6.2f}")

print(f"\n📊 میانگین خطا (با IFC واقعی): {np.mean(sanity_errors):.4f} THz")
print(f"📊 میانگین تعداد مودهای منفی: {np.mean(n_negative_modes_list):.1f} از مجموع 3×n_atoms")
print()
if np.mean(sanity_errors) > 1.0:
    print("⚠️ حتی با IFC واقعی، خطای محاسبه‌ی فرکانس بالاست!")
    print("   این یعنی مشکل از فرمول محاسبه (نقطه Γ تنها، یا نحوه‌ی ساخت ماتریس دینامیکی) است،")
    print("   نه از مدل. تقریب نقطه-Γ احتمالاً برای این مقایسه با peak کلی نوار کافی نیست —")
    print("   چون peak واقعی می‌تواند در نقاط q دیگر (نه فقط Γ) رخ دهد.")
else:
    print("✅ محاسبه از IFC واقعی منطقی است — مشکل از یادگیری مدل است، نه فرمول.")

In [ ]:
print("="*70)
print("🔬 تست ۲: توزیع eigenvalue برای یک مدل دست‌نخورده (untrained, random init)")
print("="*70)

untrained_model = DualGraphIFCNet(n_bond_features=6, n_atom_features=N_ATOM_FEATURES,
                                    edge_dim=1, max_atoms=MAX_ATOMS).to(device)
untrained_model.eval()

sample_i = test_idx[0]
s = raw_samples[sample_i]
n = s['n_atoms']

bond_batch = Batch.from_data_list([bond_graphs[sample_i]]).to(device)
atom_batch = Batch.from_data_list([atom_graphs[sample_i]]).to(device)

with torch.no_grad():
    ifc_pred_untrained = untrained_model(bond_batch, atom_batch)[0]

masses = get_masses_tensor(s['atom_elements'], MAX_ATOMS)
eigvals_untrained = compute_eigenvalues_raw(ifc_pred_untrained, masses, n)

print(f"ماده نمونه: {s['formula']} (n_atoms={n})")
print(f"Eigenvalueهای مدل دست‌نخورده (خام، قبل از ضریب تبدیل):")
print(f"   min={eigvals_untrained.min().item():.6f}, max={eigvals_untrained.max().item():.6f}")
print(f"   تعداد منفی: {(eigvals_untrained < 0).sum().item()} از {3*n}")
print()
print("مقایسه با IFC واقعی همین ماده:")
ifc_true_sample = torch.tensor(s['force_constants'], dtype=torch.float32)
eigvals_true = compute_eigenvalues_raw(ifc_true_sample, get_masses_tensor(s['atom_elements'], n), n)
print(f"   min={eigvals_true.min().item():.6f}, max={eigvals_true.max().item():.6f}")
print(f"   تعداد منفی: {(eigvals_true < 0).sum().item()} از {3*n}")
print()
print("📌 نکته: مدل دست‌نخورده باید eigenvalueهای کوچک و نزدیک صفر بدهد (چون وزن‌های شبکه تصادفی‌اند).")
print("   اگر مقیاس آن‌ها با IFC واقعی خیلی متفاوت باشد، یعنی مدل باید مسافت زیادی در فضای پارامتر طی کند.")

## 9. Physics-Informed Losses — **وزن‌های تعادل‌یافته (v3)**

### منطق بازتعادل
| Loss | v1 (ضعیف) | v2 (خیلی قوی) | **v3 (متعادل)** |
|---|---|---|---|
| λ_symmetry | 0.01 | 0.5 | **0.05** |
| λ_asr | 0.01 | 1.0 | **0.1** |
| λ_smoothness | 0.001 | 0.01 | **0.005** |

هدف: λهایی به‌اندازه‌ای بزرگ که قید فیزیکی نادیده گرفته نشود (مثل v1)، اما نه آنقدر بزرگ که IFC MSE قربانی شود (مثل v2).
این مقادیر تقریباً **میانگین هندسی** بین v1 و v2 هستند — اولین نقطه‌ی منطقی برای آزمایش.


In [ ]:
def symmetry_loss(ifc_pred, n_atoms_real):
    total = 0.0
    batch_size = ifc_pred.shape[0]
    for b in range(batch_size):
        n = n_atoms_real[b]
        ifc = ifc_pred[b, :n, :n, :, :]
        ifc_T = ifc.permute(1, 0, 3, 2)
        total = total + torch.mean((ifc - ifc_T) ** 2)
    return total / batch_size


def acoustic_sum_rule_loss(ifc_pred, n_atoms_real):
    total = 0.0
    batch_size = ifc_pred.shape[0]
    for b in range(batch_size):
        n = n_atoms_real[b]
        ifc = ifc_pred[b, :n, :n, :, :]
        asr = torch.sum(ifc, dim=1)
        total = total + torch.mean(asr ** 2)
    return total / batch_size


def smoothness_loss(frequencies_list):
    total = 0.0
    count = 0
    for freqs in frequencies_list:
        if len(freqs) >= 3:
            d2 = freqs[2:] - 2 * freqs[1:-1] + freqs[:-2]
            total = total + torch.mean(d2 ** 2)
            count += 1
    return total / max(count, 1)

print("✅ Physics losses آماده شدند")

In [ ]:
# ⚖️ وزن‌های متعادل‌شده (v3) — میانگین هندسی بین v1 (ضعیف) و v2 (قوی)
LAMBDA_SYMMETRY   = 0.05
LAMBDA_ASR        = 0.1
LAMBDA_SMOOTHNESS = 0.005


def compute_batch_loss(model, bond_data, atom_data, ifc_target_list, elements_list, device):
    ifc_pred = model(bond_data, atom_data)
    batch_size = ifc_pred.shape[0]
    n_atoms_real = torch.tensor([len(e) for e in elements_list], dtype=torch.long)

    mse_total = 0.0
    freq_preds = []
    for b in range(batch_size):
        n = n_atoms_real[b]
        pred_ifc_b = ifc_pred[b, :n, :n, :, :]
        true_ifc_b = torch.tensor(ifc_target_list[b], dtype=torch.float32, device=device)
        mse_total = mse_total + F.mse_loss(pred_ifc_b, true_ifc_b)

        masses = get_masses_tensor(elements_list[b], MAX_ATOMS).to(device)
        freq_pred_b = compute_phonon_frequencies_gamma(ifc_pred[b], masses, n)
        freq_preds.append(freq_pred_b)

    mse_total = mse_total / batch_size

    sym_loss = symmetry_loss(ifc_pred, n_atoms_real)
    asr_loss = acoustic_sum_rule_loss(ifc_pred, n_atoms_real)
    smooth_loss = smoothness_loss(freq_preds)

    total_loss = (mse_total
                  + LAMBDA_SYMMETRY * sym_loss
                  + LAMBDA_ASR * asr_loss
                  + LAMBDA_SMOOTHNESS * smooth_loss)

    return total_loss, mse_total.item(), sym_loss.item(), asr_loss.item(), smooth_loss.item(), freq_preds

print(f"✅ Loss ترکیبی با وزن‌های متعادل: λ_sym={LAMBDA_SYMMETRY}, λ_asr={LAMBDA_ASR}, λ_smooth={LAMBDA_SMOOTHNESS}")

## 10. حلقه‌ی آموزش — با لاگ سلامت Eigenvalue در هر چند Epoch

علاوه بر Loss، هر ۵۰ epoch تعداد مودهای منفی (ناپایدار) را هم روی یک batch validation چاپ می‌کنیم
تا در طول training ببینیم آیا مدل به سمت IFC فیزیکی پایدارتر حرکت می‌کند یا نه.


In [ ]:
def make_batches(indices, batch_size, shuffle=False):
    idx_arr = np.array(indices)
    if shuffle:
        idx_arr = np.random.permutation(idx_arr)
    for i in range(0, len(idx_arr), batch_size):
        yield idx_arr[i:i+batch_size]


def run_epoch(model, indices, optimizer=None, batch_size=8, shuffle=False):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_mse, total_sym, total_asr, total_smooth = 0., 0., 0., 0., 0.
    n_batches = 0

    for batch_idx in make_batches(indices, batch_size, shuffle):
        bond_list = [bond_graphs[i] for i in batch_idx]
        atom_list = [atom_graphs[i] for i in batch_idx]
        ifc_list = [ifc_targets[i] for i in batch_idx]
        elem_list = [raw_samples[i]['atom_elements'] for i in batch_idx]

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)

        with torch.set_grad_enabled(is_train):
            if is_train:
                optimizer.zero_grad()
            loss, mse_v, sym_v, asr_v, smooth_v, _ = compute_batch_loss(
                model, bond_batch, atom_batch, ifc_list, elem_list, device)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        total_loss += loss.item()
        total_mse += mse_v
        total_sym += sym_v
        total_asr += asr_v
        total_smooth += smooth_v
        n_batches += 1

    n_batches = max(n_batches, 1)
    return (total_loss/n_batches, total_mse/n_batches, total_sym/n_batches,
            total_asr/n_batches, total_smooth/n_batches)


def check_eigenvalue_health(model, indices, n_samples=10):
    """درصد مودهای منفی (ناپایدار) را روی چند نمونه برمی‌گرداند."""
    model.eval()
    total_modes, negative_modes = 0, 0
    with torch.no_grad():
        for i in indices[:n_samples]:
            s = raw_samples[i]
            n = s['n_atoms']
            bond_batch = Batch.from_data_list([bond_graphs[i]]).to(device)
            atom_batch = Batch.from_data_list([atom_graphs[i]]).to(device)
            ifc_pred = model(bond_batch, atom_batch)[0]
            masses = get_masses_tensor(s['atom_elements'], MAX_ATOMS).to(device)
            eigvals = compute_eigenvalues_raw(ifc_pred, masses, n)
            negative_modes += (eigvals < 0).sum().item()
            total_modes += len(eigvals)
    return 100.0 * negative_modes / max(total_modes, 1)

print("✅ توابع run_epoch و check_eigenvalue_health آماده شدند")

In [ ]:
model = DualGraphIFCNet(n_bond_features=6, n_atom_features=N_ATOM_FEATURES,
                         edge_dim=1, max_atoms=MAX_ATOMS).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=20)

EPOCHS = 500
PATIENCE = 100
BATCH_SIZE_TRAIN = 8

best_val_loss = float('inf')
best_state = None
patience_counter = 0
history = []

for epoch in range(EPOCHS):
    train_loss, train_mse, train_sym, train_asr, train_smooth = run_epoch(
        model, train_idx, optimizer=optimizer, batch_size=BATCH_SIZE_TRAIN, shuffle=True)

    val_loss, val_mse, val_sym, val_asr, val_smooth = run_epoch(
        model, val_idx, optimizer=None, batch_size=BATCH_SIZE_TRAIN, shuffle=False)

    scheduler.step(val_loss)

    history.append({
        'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss,
        'val_mse': val_mse, 'val_sym': val_sym, 'val_asr': val_asr, 'val_smooth': val_smooth
    })

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
        torch.save(best_state, '/kaggle/working/best_physics_informed_model_v3.pt')
    else:
        patience_counter += 1

    if epoch % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:4d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} "
              f"(MSE={val_mse:.4f}, Sym={val_sym:.4f}, ASR={val_asr:.4f}, Smooth={val_smooth:.4f}) "
              f"LR={current_lr:.6f} ⭐ Best: {best_val_loss:.4f}")

    if epoch % 50 == 0:
        neg_pct = check_eigenvalue_health(model, val_idx, n_samples=10)
        print(f"         🔬 درصد مودهای منفی (ناپایدار) در این epoch: {neg_pct:.1f}%")

    if patience_counter >= PATIENCE:
        print(f"⚠️ Early stop در epoch {epoch}")
        break

model.load_state_dict(best_state)
print(f"\n✅ آموزش کامل شد. بهترین Val Loss: {best_val_loss:.4f}")

## 11. ارزیابی نهایی روی Test و مقایسه با تمام نسخه‌ها

In [ ]:
test_loss, test_mse, test_sym, test_asr, test_smooth = run_epoch(
    model, test_idx, optimizer=None, batch_size=BATCH_SIZE_TRAIN, shuffle=False)

final_neg_pct = check_eigenvalue_health(model, test_idx, n_samples=len(test_idx))

print(f"{'='*60}")
print(f"🏁 نتیجه نهایی — مدل Physics-Informed v3 (وزن‌های متعادل)")
print(f"{'='*60}")
print(f"Test Total Loss:    {test_loss:.4f}")
print(f"Test IFC MSE:        {test_mse:.4f}")
print(f"Test Symmetry Loss:  {test_sym:.6f}")
print(f"Test ASR Loss:       {test_asr:.6f}")
print(f"Test Smoothness:     {test_smooth:.6f}")
print(f"درصد مودهای منفی:    {final_neg_pct:.1f}%")

print(f"\n{'='*60}")
print("📊 مقایسه با نسخه‌های قبلی")
print(f"{'='*60}")
print(f"{'نسخه':<10}{'IFC MSE':<12}{'ASR Loss':<12}{'MAE فرکانس':<12}")
print(f"{'Nb 11':<10}{'-':<12}{'-':<12}{'0.429':<12}")
print(f"{'v1':<10}{'0.0223':<12}{'1.860':<12}{'17.136':<12}")
print(f"{'v2':<10}{'0.1104':<12}{'0.0026':<12}{'17.276':<12}")
print(f"{'v3':<10}{test_mse:<12.4f}{test_asr:<12.4f}{'(سل بعد)':<12}")

In [ ]:
model.eval()
all_freq_mae = []

with torch.no_grad():
    for batch_idx in make_batches(test_idx, BATCH_SIZE_TRAIN, shuffle=False):
        bond_list = [bond_graphs[i] for i in batch_idx]
        atom_list = [atom_graphs[i] for i in batch_idx]
        elem_list = [raw_samples[i]['atom_elements'] for i in batch_idx]
        phonon_list = [phonon_targets[i] for i in batch_idx]

        bond_batch = Batch.from_data_list(bond_list).to(device)
        atom_batch = Batch.from_data_list(atom_list).to(device)
        ifc_pred = model(bond_batch, atom_batch)

        for b in range(len(batch_idx)):
            n = len(elem_list[b])
            masses = get_masses_tensor(elem_list[b], MAX_ATOMS).to(device)
            freq_pred = compute_phonon_frequencies_gamma(ifc_pred[b], masses, n)
            true_peak = float(np.max(phonon_list[b]))
            pred_peak = float(torch.max(torch.abs(freq_pred)).item())
            all_freq_mae.append(abs(true_peak - pred_peak))

freq_mae = np.mean(all_freq_mae)
print(f"🔬 MAE فرکانس v3 (وزن‌های متعادل، THz): {freq_mae:.4f}")
print(f"   Notebook 11 baseline: 0.429")
print(f"   v1: 17.136  |  v2: 17.276  |  v3: {freq_mae:.4f}")

if freq_mae < 0.429:
    print("\n🎉 این مدل بهتر از Notebook 11 عمل کرد!")
elif freq_mae < min(17.136, 17.276):
    print("\n📈 بهبود نسبت به v1/v2 — اما هنوز کار باقی است.")
else:
    print("\n⚠️ هنوز بهبودی کافی نیست. به نتایج سل تشخیصی (بخش ۸) مراجعه کنید:")
    print("   اگر حتی با IFC واقعی خطا بالا بود، باید روش محاسبه (تقریب نقطه-Γ به‌جای peak کلی نوار) را اصلاح کرد —")
    print("   نه اینکه فقط وزن Loss را تغییر دهیم.")

## 12. جمع‌بندی و تفسیر

### پرسش کلیدی که سل تشخیصی (بخش ۸) به آن پاسخ می‌دهد
> آیا مشکل از **روش محاسبه‌ی فرکانس** (فرمول/تقریب نقطه-Γ) است یا از **یادگیری مدل**؟

اگر سل تشخیصی نشان داد که حتی با IFC **واقعی** Phonopy، خطای محاسبه‌ی peak فرکانس بالا بود،
این یعنی **اصل ایده‌ی مقایسه‌ی "peak از نقطه Γ" با "peak کل نوار فونون در همه‌ی نقاط q" از پایه اشتباه است** —
چرا که در بسیاری از مواد، بالاترین فرکانس فونون می‌تواند در یک نقطه‌ی q دیگر (نه Γ) رخ دهد، نه اینکه مدل بد یاد گرفته باشد.

### راهکار اگر سل تشخیصی این مشکل را تأیید کرد
به‌جای مقایسه‌ی «peak از نقطه Γ» با «peak کل نوار»، باید:
1. یا فرکانس‌های نقطه Γ واقعی Phonopy (نه peak کل نوار) را به‌عنوان baseline مقایسه استخراج کرد
2. یا محاسبه را به چند نقطه‌ی q (نه فقط Γ) گسترش داد تا واقعاً قابل‌مقایسه با کل نوار باشد

### نتایج این نسخه (v3) برای ثبت در Obsidian
- IFC MSE: (بالا، از خروجی سل ۱۲)
- ASR Loss: (بالا)
- MAE فرکانس: (بالا)
- نتیجه‌ی سل تشخیصی: (آیا مشکل از فرمول بود یا از مدل)
